# 08 · Transiciones de recepción y montos pensionarios, ENASEM

**Pregunta central conservada:** ¿Qué tanto es consistente la expansión de las pensiones y los apoyos públicos con una reducción de las brechas de cobertura y de monto de la protección económica de los adultos mayores en México (ENASEM 2001–2021), qué grupos siguen sin vía propia de protección y cómo cambia esta lectura al considerar a quienes mueren o dejan de ser entrevistados?

**Objetivo independiente:** explicar la divergencia cobertura/monto mediante recepción observada, nuevos y continuos, precios, composición y selección. Se profundiza en K 2012–2015 y A general 2015–2018; no se repite el EDA ni se modifica 01–07.

Principal: cohorte basal de personas vivas registradas con entrevista directa, 60+ por visita inicial. No panel balanceado, causalidad, ponderación nacional ni identidad administrativa del beneficio. Los conflictos de estado vital se conservan y separan de las transiciones válidas.

No se suman importes cuando algún evento captado carece de monto. Captura íntegra no garantiza todos los beneficios reales. A incluye dinero/especie; **A5 de vejez no es una serie longitudinal en estos periodos**. [Contrato de definiciones08](../diccionarios/contrato_transiciones_montos_pensionarios.json).

[Copia ejecutada con resultados](../data/interim/notebooks_ejecutados/08_TRANSICIONES_MONTOS_PENSIONARIOS_ejecutado.ipynb). [Informe académico único](../resultados/INFORME_GENERAL.md).

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, Image
from tesis_enasem.benefit_transitions import analyze, OUTPUT, verify_inputs
from tesis_enasem.benefit_transition_outputs import interpretations
from tesis_enasem.project_paths import RESULTS_DIR
verify_inputs()
r=analyze()  # ejecuta08 desde cero; no ejecuta01–07
assert r==json.loads((OUTPUT/'resultados.json').read_text()),'08 no reproducido exactamente'
audit=r['auditoria'];texts=interpretations(r)
figs=json.loads((OUTPUT/'figuras.json').read_text())
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',80)
pd.set_option('display.float_format',lambda x:f'{x:,.3f}')
def tabla(rows):
    display(pd.json_normalize(rows,sep='.'))
def figura(key):
    f=next(f for f in figs if f['clave']==key)
    display(Image(filename=str(RESULTS_DIR/f['archivo'])))
    display(Markdown('**Nota:** '+f['nota']))
def contexto(z):return dict(origen=z['origen'],destino=z['destino'],via=z['via'])
def montos(rows):
    return [dict(**{k:v for k,v in z.items() if k not in ['nominal','real','real_12m','real_ano_entrevista']},
        nominal_N=z['nominal']['N'],nominal_mediana=z['nominal']['mediana'],
        nominal_q25=z['nominal']['q25'],nominal_q75=z['nominal']['q75'],
        real_N=z['real']['N'],real_mediana=z['real']['mediana'],
        real_q25=z['real']['q25'],real_q75=z['real']['q75']) for z in rows]


## 1. Auditoría de fuentes y seis rondas

Antes de comparar montos: población, recepción, integridad de captura y unidad.

In [ ]:
tabla(audit['rondas']); tabla(audit['intervalos']); tabla(audit['comparabilidad']); tabla([{'via':k,**v} for k,v in audit['diagnosticos']['eventos'].items()])

**Interpretación, estado y límites:** Se reutilizan recepción, titularidad, eventos y precios de 01–07. Los 1,762 pares K y 1,240 pares A de 07 admitían sumas parciales. Al exigir monto de cada evento captado, fecha válida y ausencia de conflicto vital quedan 1,756 y 1,223. En A hay un par con entrevista viva posterior a registro sobre fallecido: se separa sin corregir el original; 1,224 queda solo como sensibilidad de registros observados. Completo significa íntegro respecto de captura, no totalidad de beneficios reales. No hay identificador administrativo de prestación. K cambia de un importe por contingencia en 2012 a slots en 2015; A pasa de importe agregado a eventos en 2018. La comparabilidad de la suma por persona es condicionada. A es donativo general en dinero o especie; no A5 de vejez, identificado solo en 2021.

## 2. Transiciones de recepción en las seis ventanas elegibles

Mismos IDs, LEFT uno:uno y denominador de toda la cohorte o pares conocidos explícito.

In [ ]:
tabla([{**contexto(z),**x,'N_cohorte':z['N_cohorte'],'N_pares_conocidos':z['N_pares_conocidos']} for z in r['transiciones'] for x in z['estados']]); figura('seguimiento')

**Interpretación, estado y límites:** No→Sí es inicio observado de respuesta afirmativa; Sí→No es fin observado de declaración. No alta/baja administrativa. Muerte, ausencia, sustituto, recepciónND y conflicto vital son categorías distintas. Las rupturas de K/A quedan excluidas de las ventanas analizadas.

## 3. Denominadores y cambios de universo

Distinguir probabilidad condicional a reobservación de fracción sobre cohorte basal.

In [ ]:
tabla([{**contexto(z),**{k:v for k,v in z.items() if k not in ['estados','origen','destino','via']}} for z in r['transiciones']]); tabla([{**contexto(z),**z['otra_via_previa']} for z in r['analisis']])

**Interpretación, estado y límites:** Nuevos K pueden haber recibido A antes, y nuevos A pueden haber tenido K. No equivalen a primera protección económica. Las tasas condicionales excluyen muertes/ausencias y no representan riesgos de toda la cohorte.

## 4. Nuevos frente a continuos y cese de declaración

Importes de destino para nuevos/continuos; importes basales para Sí→No.

In [ ]:
tabla([{**contexto(z),**x} for z in r['analisis'] for x in montos(z['montos_grupos'])]); figura('nuevos_continuos')

**Interpretación, estado y límites:** En K los nuevos observados tienen mediana mensual $580 nominales/$992.86 reales (N válido 1,435/1,469); los continuos, $2,500/$4,279.56 (N 1,857/1,975). La diferencia nuevos−continuos es $-3,286.70 reales; la mediana nueva equivale a 23.20% de la continua. Es un contraste sustantivo entre receptores observados, no entre iguales ni de toda la cohorte. En A ambos grupos tienen mediana de $580/$886.48, con N 1,173/1,212 y 1,245/1,281. Su Q 75 real sí difiere ($1,451.99 frente a $1,528.41); la mediana nula no implica distribuciones idénticas. El monto de continuos en destino no exige monto basal válido; los cambios individuales sí. No se suman importes con algún evento faltante.

## 5. Mezcla observada y entradas a la población60+

No confundir nuevos receptores de la cohorte con muestra nueva, reaparición o entrada por edad.

In [ ]:
tabla([{**contexto(z),**x} for z in r['analisis'] for x in z['mezcla_observada']]); tabla([{**contexto(z),**x} for z in r['analisis'] for x in z['composicion_destino']['grupos']]); tabla([{**contexto(z),**x} for z in r['analisis'] for x in z['perfiles_transversales06_07']])

**Interpretación, estado y límites:** Entre 3,292 receptores K de la cohorte basal con monto íntegro de destino, la mezcla observada de nuevos y continuos tiene mediana $2,100/$3,594.83; los 1,857 continuos solos, $2,500/$4,279.56. Incluir el grupo nuevo de menor distribución reduce aquí la mediana empírica del conjunto seleccionado. No cuantifica una fracción causal de la caída transversal de 11.05%: excluye entradas por edad, otros orígenes y casos sin monto. En A, la mezcla de 2,418 y los 1,245 continuos solos mantienen la misma mediana $886.48. Es evidencia contraria a una explicación general por menor mediana de nuevos. Una mediana de mezcla no es promedio ponderado de las medianas de sus grupos.

K 2012–2015 (N receptoras 2,819/4,159): Mujer 42.89%→46.89%; 70+ 52.93%→59.27%; 0 años 14.44%→17.14%; <100 mil habitantes 28.70%→39.24%. A 2015–2018 (N receptoras 2,630/2,764): Mujer 59.32%→62.66%; 70+ 65.86%→73.99%; 0 años 28.97%→28.51%; <100 mil habitantes 59.58%→61.40%. Son porcentajes dentro de todos los receptores propios directos 60+ de cada ronda, con denominadores de receptoras, no tasas de cobertura. Se reutilizan los conteos de 06/07. No describe las mismas personas ni separa incorporación, envejecimiento, nueva muestra y muerte. La comparación de patrones y la partición de destino documentan algunos componentes observables, sin una descomposición causal ni completa de la mediana. Un registro con antecedente sobre fallecido se conserva como conflicto, sin afirmar pago posterior a muerte.

## 6. Instituciones y cambios de concepto K

Comprobar evidencia, sin asignar el mismo programa a toda K o repartir importes multiinstitucionales.

In [ ]:
tabla([{**contexto(z),**x} for z in r['analisis'] for x in z['evidencia_institucional']])

**Interpretación, estado y límites:** Código8 agrupa programas públicos federales/estatales. IMSS/ISSSTE identifica evidencia institucional, no contributividad. Las categorías pueden solaparse; los importes son personales de todos los eventos íntegros, no monto atribuible a cada institución.

## 7. Tres estimandos monetarios distintos

Medianas transversales, diferencias de medianas de pares y medianas de cambios personales.

In [ ]:
tabla([{**contexto(z),'ronda':x['ronda'],'N_parcial07':x['N_monto_parcial07'],'nominal_parcial':x['nominal_parcial07']['mediana'],'real_parcial':x['real_parcial07']['mediana'],'N_integro':x['real_integro']['N'],'nominal_integro':x['nominal_integro']['mediana'],'real_integro':x['real_integro']['mediana']} for z in r['analisis'] for x in z['transversal']]); tabla([{**contexto(z),**z['pares']} for z in r['analisis']]); figura('cambios_personales')

**Interpretación, estado y límites:** En 1,756 continuos K con ambas capturas íntegras, la mediana nominal pasa de $2,200 a $2,500; las medianas reales son $4,233.66 y $4,279.56: diferencia +$45.89. La mediana del cambio individual es $-40.26; 932/1,756 (53.08%) muestran reducción. No son resultados incompatibles: cambian los rangos relativos de las personas dentro de la distribución. En A, N=1,223, las medianas nominales son $550/$580, y las reales $941.50/$886.48. La diferencia de medianas y la mediana individual coinciden aquí en $-55.02, por esta distribución, no por identidad estadística. Bajan 736/1,223 (60.18%). Existen aumentos y disminuciones en ambos casos; no todos los receptores pierden poder adquisitivo.

## 8. Inflación y nominal: identidad contable

Aplicar INPC por entrevista real antes de resumir; no confundir fecha elegida y fecha de pago.

In [ ]:
tabla([{**contexto(z),**z['pares']['puente_medianas']} for z in r['analisis']]); tabla([{**contexto(z),**z['pares']['identidad_contable']} for z in r['analisis']]); tabla([{**contexto(z),'inflacion':z['pares']['inflacion_criterio_periodo_pct'],'meses':z['pares']['meses_entrevistas']} for z in r['analisis']])

**Interpretación, estado y límites:** Se mantiene INPC oficial INEGI 146.075, septiembre de 2026. Para cada persona, `R1−R0 = f1×(M1−M0) + M0×(f1−f0)`, donde `ft=146.075/INPCt`. El primer término valora el cambio nominal a precios de destino y el segundo congela el nominal basal. La identidad es exacta, no una estimación de mecanismos causales. Las medianas de esos términos no se suman como medianas de cambios. Un puente adicional de medianas, sobre los mismos pares, primero repricia todos los nominales basales y luego los sustituye por los finales: K: $-467.65 + $+513.55 = $+45.89; A: $-100.88 + $+45.85 = $-55.02. El orden inverso, documentado en los agregados, produce componentes distintos. No se interpreta como porcentaje de inflación frente a composición en la mediana transversal. La inflación mediana bajo el criterio del año calendario previo es 12.42% en K y 12.00% en A. La comparación modal utiliza 2011→2014 y 2014→2017; entrevistas tardías conservan sus índices individuales. Son intervalos de precios elegidos para año pasado, no fechas de pago observadas ni inflación medida exactamente entre las visitas. Bajo un factor común, `1+cambio_real = (1+cambio_nominal)/(1+inflacion)`: 0% nominal y 12.42% de inflación producen −11.05%; +5.45% y 12.00% producen −5.84%. Esta aritmética explica la traducción monetaria, pero no por qué la distribución nominal quedó en esas medianas.

## 9. Sensibilidad temporal, de evento y concepto

Revisar magnitudes pequeñas y signos dependientes del calendario.

In [ ]:
tabla([{**contexto(z),**{k:x.get(k) for k in ['criterio','N','N_vital_discordante']},'delta_real':x['cambio_individual_real']['mediana'],'delta_12m':x['cambio_real_12m']['mediana'],'delta_ano_entrevista':x['cambio_real_ano_entrevista']['mediana']} for z in r['analisis'] for x in z['sensibilidades'] if 'cambio_individual_real' in x])

**Interpretación, estado y límites:** La mediana individual K es pequeña y sensible al periodo: $-13.59 con doce meses previos y $+5.86 con año de entrevista, frente a $-40.26 principal. No se sostiene una caída individual mediana robusta de K a todos los criterios temporales. La restricción de contingencia e institución también cambia el universo: no certifica continuidad administrativa ni contributividad. A mantiene signo negativo con $-68.17/$-71.49. La sensibilidad de un evento por visita no identifica que sea el mismo programa A. Ninguna sensibilidad garantiza invariancia de captura.

## 10. Perfiles de nuevos, continuos y cese

Sexo, edad, escolaridad inicial, localidad, ocupación y estado conyugal: basales y finales separados.

In [ ]:
for z in r['analisis']:
    display(Markdown(f"### {z['via']} {z['origen']}–{z['destino']}"))
    for dim in ['sexo_grupo','edad_06','educacion_grupo','localidad_grupo','ocupacion_grupo','conyugal_grupo']:
        tabla([x for x in z['perfiles'] if x['dimension']==dim])

**Interpretación, estado y límites:** **K perfiles basales:** Mujer: 54.87% nuevos, 40.46% continuos, 58.20% cese; 70+: 44.93% nuevos, 48.10% continuos, 59.16% cese; 0 años: 26.34% nuevos, 12.05% continuos, 26.37% cese; <100 mil habitantes: 54.46% nuevos, 27.85% continuos, 34.08% cese; Trabaja: 31.65% nuevos, 21.11% continuos, 18.97% cese; Viudo: 19.61% nuevos, 32.91% continuos, 27.97% cese. Los N totales de cada patrón son 1469/1975/311.

**A perfiles basales:** Mujer: 61.39% nuevos, 64.01% continuos, 55.06% cese; 70+: 52.39% nuevos, 63.62% continuos, 61.17% cese; 0 años: 26.65% nuevos, 30.44% continuos, 22.61% cese; <100 mil habitantes: 57.43% nuevos, 62.92% continuos, 50.90% cese; Trabaja: 30.45% nuevos, 29.74% continuos, 30.93% cese; Viudo: 22.61% nuevos, 28.42% continuos, 24.41% cese. Los N totales de cada patrón son 1212/1281/721.

Estos porcentajes usan toda la transición conocida, incluyendo personas sin monto íntegro; son composición del grupo, no cobertura de cada perfil. Las personas con variables desconocidas permanecen en el denominador total y se identifican en las tablas completas. Los nuevos K no son simplemente más viejos: tienen mayor presencia femenina, menor escolaridad y localidades pequeñas, pero menor proporción basal 70+ que continuos. El acceso por una vía y el monto central entre receptores describen desigualdades diferentes. En nuevos K, escolaridad 0 tiene mediana $550/$941.50 (N=385), frente a ≥10 años $6,000/$10,270.94 (N=83). Localidad<100 mil: $550/$941.50 (N=791); ≥100 mil: $1,350/$2,310.96 (N=644). La diferencia K nuevos/continuos se observa dentro de ambos sexos y los perfiles seleccionados, aunque su magnitud varía. En A, ambos sexos mantienen la misma mediana general entre nuevos y continuos. Entre personas basales 60–64 aparece el contraste inverso: nuevos $886.48 (N=145) frente a continuos $726.00 (N=99). Grupos basales de edad no equivalen a elegibilidad del programa al destino. Las diferencias de cambios personales son pequeñas, heterogéneas y no pruebas de efectos. La escolaridad es inicial; el corte 100 mil no define ruralidad; ocupación 4 no identifica jubilación efectiva; viudez no acredita pareja residente.

## 11. Montos y cambios personales por perfil

Cuantiles monetarios en ambas unidades y medianas de cambios, con N de cada población.

In [ ]:
for z in r['analisis']:
    display(Markdown(f"### {z['via']} {z['origen']}–{z['destino']}"))
    tabla(z['montos_por_perfil'])
    tabla([{k:x[k] for k in ['dimension','grupo','N','nominal_inicial','nominal_final','real_inicial','real_final','cambio_individual_nominal','cambio_individual_real','N_real_baja','N_real_sube']} for x in z['cambios_por_perfil']])

**Interpretación, estado y límites:** Medianas/IQR/P10/P90 reducen el dominio de extremos; las medias no gobiernan la interpretación. No se eliminan importes altos ni se fabrican Gini con cuantiles. Grupos pequeños y selección limitan la precisión; no pruebas de significancia o inferencia nacional.

## 12. Comparación por composición común

Estimando explícito y celdas mínimas, con soporte y cobertura por grupo.

In [ ]:
tabla([{**contexto(z),**{k:v for k,v in z['ajuste'].items() if k not in ['estratos','distribuciones']}} for z in r['analisis']]); tabla([{**contexto(z),**x} for z in r['analisis'] for x in z['ajuste']['estratos']]); tabla([{**contexto(z),**x} for z in r['analisis'] for x in z['ajuste']['distribuciones']])

**Interpretación, estado y límites:** Se estandarizan las distribuciones de destino a una mezcla común de sexo, edad basal (60–64, 65–69, 70+) y escolaridad inicial (0–5/6+). Cada celda exige al menos 20 nuevos y 20 continuos; se requiere cubrir ≥70% de cada grupo. La mezcla objetivo es la composición conjunta empírica dentro del soporte. Los cuantiles se obtienen de la CDF ponderada, no promediando medianas de celdas. El comparador sin ajuste usa la misma definición de cuantil y el mismo soporte. K conserva 1,428/1,435 nuevos y 1,846/1,857 continuos; las medianas reales ajustadas son $992.86 y $4,108.38. El contraste de K persiste después de equilibrar esa composición basal. A conserva 1,087/1,173 y 1,192/1,245; las medianas ajustadas siguen iguales. Añadir localidad tiene soporte suficiente y no elimina la lectura principal. Ocupación y estado conyugal se describen, sin asegurar equilibrio conjunto. Los pesos son de composición descriptiva, no de encuesta ni de probabilidad de observación. No se corrige selección no observada, mortalidad ni causalidad. No se implementa Oaxaca–Blinder o Kitagawa.

## 13. Localidad en soporte y modalidad por sustituto

Sensibilidades con sus propios N, no sustitución del principal.

In [ ]:
tabla([{**contexto(z),**{k:v for k,v in z['ajuste_localidad_sensibilidad'].items() if k not in ['estratos','distribuciones']}} for z in r['analisis']]); tabla([{**contexto(z),**x} for z in r['analisis'] for x in z['ajuste_localidad_sensibilidad']['distribuciones']]); tabla([{**contexto(z),**x} for z in r['analisis'] for x in z['sensibilidades'] if 'patron' in x])

**Interpretación, estado y límites:** La sensibilidad permite sustituto válido solo en destino manteniendo las mismas directas basales. No equivale a incluir sustitutos basales. La estabilidad de medianas generales no garantiza ausencia de selección ni igualdad de las colas.

## 14. Mortalidad, ausencia y reapariciones

Características y montos basales de todos los destinos; no imputación de pagos post mortem.

In [ ]:
tabla([{**contexto(z),'recepcion_basal':s['recepcion_basal'],'N_basal':s['N_basal'],**x} for z in r['analisis'] for s in z['seleccion'] for x in s['destinos']]); tabla([{**contexto(z),**{k:v for k,v in s.items() if k!='destinos'}} for z in r['analisis'] for s in z['seleccion']])

**Interpretación, estado y límites:** K 2012–2015: muertes registradas 464/5,919 (7.84%) en negativas iniciales y 249/2,819 (8.83%) en receptoras; ausencias 226 (3.82%) y 157 (5.57%). A 2015–2018: muertes registradas 467/6,883 (6.78%) en negativas iniciales y 282/2,630 (10.72%) en receptoras; ausencias 758 (11.01%) y 147 (5.59%). La dirección de la diferencia de ausencia cambia entre vías. Son asociaciones basales sin ajustar por edad/composición, no efectos pensionarios sobre supervivencia o respuesta. Solo los sobrevivientes reentrevistados con recepción y montos válidos pueden clasificarse como nuevos/continuos para comparar dinero. No se identifica la trayectoria monetaria de toda la cohorte. Las medianas basales similares en ciertos destinos no garantizan retención ignorable: las colas y perfiles pueden variar. Las reapariciones demuestran que ausencia no es muerte; un fallecimiento registrado después no fecha el deceso en la visita ausente. La estandarización de observados no soluciona esta selección. Permitir sustituto válido en destino conserva las medianas generales de nuevos/continuos K y A, con N mayores, pero no prueba ausencia de sesgo en subgrupos. Ningún muerto recibe un monto posterior imputado.

## 15. Política vigente, indicador y lectura

**PAM 65+ desde 2013, $580 mensuales desde 2014 → indicador K propio y evidencia código 8 → 927/1,435 nuevos con monto íntegro (64.60%) tienen evidencia de programa público agrupado, frente a 144/1,857 continuos (7.75%) → límite:** código 8 incluye programas federales/estatales y puede coexistir con otras instituciones; no identifica PAM ni permite asignarle toda la suma. Entre personas con evidencia IMSS/ISSSTE, nuevos (N=433) y continuos (N=1,671) tienen la misma mediana personal nominal $2,500; los nuevos con programa agrupado (N=927) tienen mediana $550. Estas sumas personales incluyen todos los eventos íntegros, no montos asignados a cada institución; las evidencias pueden solaparse. Esto documenta heterogeneidad de la incorporación captada, sin atribución causal. 

**PAM mantiene $580 entre 2014–2018 → indicador A general y norma por separado → mediana A pasa 550→580 nominales, mientras precios del criterio principal aumentan aproximadamente 12% y la mediana real cae → límite:** A no identifica PAM en ambas visitas, incluye dinero/especie y cambia captación en 2018. No son una serie individual de monto oficial ni todo el padrón. Las coincidencias son compatibles con expansión de prestaciones pequeñas y erosión normativa, sin demostrar que una reforma produjo el resultado. Se usa la [cronología y sus fuentes oficiales](../investigacion/notas/politicas_publicas_envejecimiento.md) de 07.

## 16. Conclusión integrada y límites

La divergencia entre cobertura y monto central no tiene una explicación única. En K 2012–2015 se observa una incorporación declarada con montos mucho menores que los de continuos, además de erosión por precios. En A 2015–2018 las medianas de nuevos y continuos coinciden, y la caída real también aparece siguiendo a las mismas personas. No se identifica que nuevos receptores con menores medianas expliquen el caso A. La selección, la captación y la composición limitan la generalización.

**Respuesta a los mecanismos.** La inflación aporta una erosión aritmética verificable bajo fechas explícitas. En K también se observa una diferencia monetaria grande de incorporación y una mezcla institucional diferente, persistente bajo estandarización basal. En A la menor mediana de nuevos no está respaldada; hay diferencias de colas y perfiles, pero la reducción real central también aparece entre continuos. La combinación observable varía por vía. No se puede repartir la caída transversal en porcentajes causales de precios, composición y política. Tampoco se pueden certificar flujos de una misma prestación, suficiencia total o ingreso individual completo. El aporte a la tesis es mostrar por qué expandir recepción no implica igualar recursos centrales y distinguir una expansión con montos menores de una erosión entre continuos. La selección delimita a quién representan las comparaciones.

**Estado:** validado aritméticamente y reproducido desde cero; comparabilidad operacional condicionada, identidad de programa y representatividad no acreditadas. Resultados relevantes en el mismo informe y PDF.

In [ ]:
verify_inputs()
assert r==json.loads((OUTPUT/'resultados.json').read_text())
print('08 reproducido, denominadores y fuentes anteriores verificados.')